# classifier

Executable notebook module. English explanations precede each functional block. Restart the kernel after changing modules.


In [ ]:
from pathlib import Path
import json, sys, types
PROJECT = Path.cwd()
assert (PROJECT / 'app.ipynb').is_file(), 'Open this notebook from EcoSort-Demo.'

def load_notebook(name):
    if name in sys.modules:
        return sys.modules[name]
    path = PROJECT / (name + '.ipynb')
    notebook = json.loads(path.read_text())
    module = types.ModuleType(name)
    module.__file__ = str(path)
    sys.modules[name] = module
    try:
        for dependency in notebook['metadata'].get('dependencies', []):
            load_notebook(dependency)
        for cell in notebook['cells']:
            if cell['cell_type'] == 'code' and 'module' in cell.get('metadata', {}).get('tags', []):
                exec(compile(''.join(cell['source']), str(path), 'exec'), module.__dict__)
    except Exception:
        sys.modules.pop(name, None)
        raise
    return module

load_notebook('disposal_rules')
__file__ = str(PROJECT / 'classifier.ipynb')


## Imports and configuration

Load required libraries and define the paths, labels and constants used by the following functions.


In [ ]:
"""Project-trained MobileNetV3; no image-text or zero-shot fallback."""
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import json
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps
import tensorflow as tf
from disposal_rules import disposal_guidance

BASE = Path(__file__).resolve().parent
RESULTS = BASE / "model"




## TrainedPredictor

Group related state and operations for this module.

- `__init__`: Initialize stored settings and model state.
- `probabilities`: Preprocess a photo and normalize temperature-scaled model scores.
- `identify`: Select a candidate and apply the uncertainty review policy.


In [ ]:
class TrainedPredictor:
    def __init__(self):
        completed = json.loads((RESULTS / "results.json").read_text())["training"]
        assert completed["status"] == "completed"
        self.settings = json.loads((RESULTS / "settings.json").read_text())
        self.classes = self.settings["classes"]  # Preserve the exact output-label order used during training.
        assert self.settings["training_images"] == completed["split_counts"]["train"]
        assert completed["verified_image_count"] == sum(completed["split_counts"].values())
        self.model = tf.keras.models.load_model(RESULTS / "model.keras", compile=False)  # Load weights for inference without rebuilding training optimizer state.
        assert self.model.output_shape[-1] == len(self.classes) == 17  # Require one output for each of the 17 classes.

    def probabilities(self, image):
        rgb = ImageOps.exif_transpose(image).convert("RGB").resize((224, 224), Image.Resampling.BILINEAR)  # Correct EXIF rotation, convert to three channels and resize consistently.
        raw = self.model(np.asarray(rgb, dtype=np.float32)[None, ...], training=False).numpy()[0]  # Add a batch dimension; disable training behavior; take the first prediction.
        logits = np.log(np.clip(raw, 1e-8, 1)) / self.settings["temperature"]  # Clip before log to avoid log(0), then apply validation-fitted temperature.
        scores = np.exp(logits - logits.max())  # Subtract the maximum to make exponentiation numerically stable.
        scores /= scores.sum()  # Normalize the scores so they sum to one.
        assert scores.shape == (len(self.classes),) and np.isfinite(scores).all()
        return scores

    def identify(self, image, food_service=False):
        scores = self.probabilities(image)
        order = np.argsort(scores)
        label = self.classes[int(order[-1])]
        uncertain = bool(scores[order[-1]] < self.settings["review_threshold"]  # Request review if the highest score is below the selected threshold.
                         or scores[order[-1]] - scores[order[-2]] < self.settings["minimum_top_two_margin"]  # Also review when the top two candidates are too close.
                         or label in self.settings["manual_review_classes"])  # Always review designated ambiguous or sensitive categories.
        # Material-only training cannot establish bottle, bag or container identity.
        route, guidance = disposal_guidance(label, food_service=food_service, uncertain=uncertain)  # Ask the rule module for guidance without inventing object identity.
        return dict(label=label, object=label.replace("_", " "), uncertain=uncertain,
                    route=route, guidance=guidance, scores=dict(zip(self.classes, map(float, scores))))
